In [4]:
import pandas as pd
from datasets import Dataset

# 1. Load the generated CSV dataset
df = pd.read_csv("data/dataset.csv", skip_blank_lines=False)

sentences = []
tags = []
current_sentence = []
current_tags = []

# 2. Group words back into full sentences
for _, row in df.iterrows():
    if pd.isna(row['Token']):  # Blank line means a new sentence starts
        if current_sentence:
            sentences.append(current_sentence)
            tags.append(current_tags)
            current_sentence = []
            current_tags = []
    else:
        current_sentence.append(str(row['Token']))
        current_tags.append(str(row['Tag']))

# 3. Map string tags to numeric IDs for the Neural Network
tag2id = {'O': 0, 'B-AR': 1, 'I-AR': 2, 'B-FR': 3, 'I-FR': 4, 'B-EN': 5, 'I-EN': 6}
numeric_tags = [[tag2id[tag] for tag in tag_seq] for tag_seq in tags]

# 4. Package into a Hugging Face Dataset format
my_dataset = Dataset.from_dict({
    "tokens": sentences,
    "ner_tags": numeric_tags
})

print(f"Success! Loaded {len(my_dataset)} mixed sentences into memory.")
print("Example sentence tokens:", my_dataset[0]['tokens'])
print("Example sentence tags:", my_dataset[0]['ner_tags'])

Success! Loaded 1920 mixed sentences into memory.
Example sentence tokens: ['salaam', 'kif', 'dayr', 'you', 'are', 'ready', 'pour', 'ce', 'soir', '?']
Example sentence tags: [1, 2, 2, 5, 6, 6, 3, 4, 4, 0]


In [6]:
import torch
import torch_directml
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

# 1. Setup the AMD GPU Setup
dml_device = torch_directml.device()
print(f"Engine starting on: {dml_device} (AMD GPU)")

# 2. Download and Load the Base AI Brain (XLM-RoBERTa)
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=7)

# Physically move the AI into the AMD Graphics Card's memory
model.to(dml_device) 

# 3. The Alignment Function (Crucial for Code-Switching)
# This prevents the AI from crashing when French/Arabizi words are split into sub-tokens
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Tell PyTorch to ignore this during loss calculation
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100) 
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 4. Process the dataset through the alignment function
print("Tokenizing the Moroccan code-switching data...")
tokenized_datasets = my_dataset.map(tokenize_and_align_labels, batched=True)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 5. Training Rules (Optimized for 16GB VRAM)
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16, # High batch size for fast training
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=10
)

# 6. Initialize the Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# 7. Start the Heavy Lifting!
print("Firing up the GPU. Training starting...")
trainer.train()

# 8. Save the final model for your web app
trainer.save_model("./saved_model")
print("Training complete! Model securely saved to the ./saved_model folder.")

Engine starting on: privateuseone:0 (AMD GPU)


c:\Projects\code_switching_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizing the Moroccan code-switching data...


Map: 100%|██████████| 1920/1920 [00:00<00:00, 20907.60 examples/s]
c:\Projects\code_switching_project\.venv\Lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Firing up the GPU. Training starting...


...

{'loss': 0.1655, 'grad_norm': 6.438869953155518, 'learning_rate': 8.333333333333333e-07, 'epoch': 3.83}


 98%|█████████▊| 470/480 [31:20<00:43,  4.34s/it]

{'loss': 0.1751, 'grad_norm': 4.756875514984131, 'learning_rate': 4.1666666666666667e-07, 'epoch': 3.92}


100%|██████████| 480/480 [31:59<00:00,  4.00s/it]


{'loss': 0.1595, 'grad_norm': 4.911510944366455, 'learning_rate': 0.0, 'epoch': 4.0}
{'train_runtime': 1919.3902, 'train_samples_per_second': 4.001, 'train_steps_per_second': 0.25, 'train_loss': 0.4181637428700924, 'epoch': 4.0}
Training complete! Model securely saved to the ./saved_model folder.
